In [ ]:
import numpy as np
import onnxruntime
import re
from transformers import AutoTokenizer
from typing import List, Dict, Tuple, Optional
import nltk
import os

In [ ]:
import nltk
import threading
from typing import Optional, Set

def get_russian_stopwords_with_timeout(timeout_seconds: float = 3.0) -> Optional[Set[str]]:
    """Пытается загрузить русские стоп-слова NLTK с заданным таймаутом."""

    # Контейнер для результата, передаваемый между потоками
    result = {'stop_words': None}

    def download_task():
        nonlocal result
        try:
            # Сначала проверяем, есть ли файлы локально (быстро)
            result['stop_words'] = set(nltk.corpus.stopwords.words('russian'))
        except LookupError:
            # Если нет, пытаемся скачать (сетевой вызов, который может зависнуть)
            try:
                nltk.download('stopwords', quiet=True)
                result['stop_words'] = set(nltk.corpus.stopwords.words('russian'))
            except Exception:
                pass # Оставляем None, если не удалось

    download_thread = threading.Thread(target=download_task)
    download_thread.start()

    # Блокируем главный поток, но ждем не более timeout_seconds
    download_thread.join(timeout=timeout_seconds)

    if download_thread.is_alive():
        # Если поток жив, значит, произошел таймаут
        # Мы не можем принудительно остановить поток, но можем проигнорировать его результат
        print(f"❌ NLTK download timed out after {timeout_seconds}s.")
        return None

    if result['stop_words'] is None:
        # Если поток завершился, но stop_words = None, значит, была ошибка
        print("NLTK download failed.")
        return None

    return result['stop_words']

In [ ]:
def classify_names(
    names_list: List[str],
    max_length: int = 32
) -> Tuple[Optional[List[int]], Optional[List[str]]]:


    ONNX_FILE = "rubert_tiny2_seq_class.onnx"
    TOKENIZER_FOLDER = 'ml_data/bert_model'

    try:
        nltk.download('stopwords', quiet=True)
        stop_words = set(nltk.corpus.stopwords.words('russian'))
    except Exception:
        stop_words = None

    def preprocess_inputs(names: List[str]) -> List[str]:
        processed_names = list(names)
        for i, name in enumerate(processed_names):
            text = name
            text = re.sub(r"«.*»", "", text)
            text = re.sub(r"([(][^)]+[)])|([\d]+\s?[в]\s?[\d+]|[№][\d]+)", "", text)
            text = re.sub(r"([\d]+[.,]?[\d]+[%])|([\d]+[%])|(\d+[+])", "", text)
            text = re.sub(r"\b\d+\s?(г|гр|гр\.|мл|л|шт|уп|пакет|%)\b", "", text, flags=re.IGNORECASE)
            text = re.sub(r"[ ]{2,}", " ", text)
            if stop_words is not None:
                words = [w for w in text.lower().split() if w not in stop_words]
                text = " ".join(words)
            processed_names[i] = text.strip()
        return processed_names

    new_names_list = preprocess_inputs(names_list)

    try:
        tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_FOLDER)
    except Exception as e:
        print(f"Ошибка загрузки токенайзера из {TOKENIZER_FOLDER}. {e}")
        return None, new_names_list

    def tokenize_inputs(texts: List[str]) -> Dict[str, np.ndarray]:
        encoded_inputs = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='np'
        )
        input_ids = encoded_inputs['input_ids'].astype(np.int64)
        attention_mask = encoded_inputs['attention_mask'].astype(np.int64)
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask
        }

    tokenized_data = tokenize_inputs(new_names_list)

    try:
        session = onnxruntime.InferenceSession(os.path.join(TOKENIZER_FOLDER, ONNX_FILE))
    except Exception as e:
        print(f"Ошибка при загрузке ONNX модели из {ONNX_FILE}. {e}")
        return None, new_names_list

    input_feed = {
        'input_ids': tokenized_data['input_ids'],
        'attention_mask': tokenized_data['attention_mask']
    }

    raw_outputs = session.run(None, input_feed)
    logits = raw_outputs[0]
    predictions = np.argmax(logits, axis=1).tolist()

    return predictions, new_names_list